In [15]:
import numpy as np

X = np.load('mnist_test_seq.npy')
X = X[:, :, np.newaxis, :, :]
X.shape

(20, 10000, 1, 64, 64)

In [16]:
X[0, 0, :, :]

array([[[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]]], shape=(1, 64, 64), dtype=uint8)

In [17]:
from torch.utils.data import DataLoader, Dataset, random_split
import torch
X = torch.from_numpy(X).float()
class MovingObjectsDataset(Dataset):
    def __init__(self, X):
        self.X = X

    def __len__(self):
        return self.X.shape[1]

    def __getitem__(self, idx):
        return self.create_shifted_frames(idx)

    def create_shifted_frames(self, idx):
        #print(self.X.shape)
        sample = self.X[:, idx, :, :]
        return sample

dataset = MovingObjectsDataset(X)
dataloader = DataLoader(dataset, batch_size=1)

#next(iter(dataloader)).size()

In [18]:
X.shape[1]

10000

In [19]:
tt_split = 0.8
train_dataset, test_dataset = random_split(
    dataset,
    [int(len(dataset) *tt_split), len(dataset) - int(len(dataset) * tt_split)])

#train_dataloader = DataLoader(train_dataset, batch_size=32)
#test_dataloader = DataLoader(test_dataset)

#next(iter(train_dataloader)).size()

In [20]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [21]:
class ConvLSTMCell(nn.Module):
    def __init__(self, input_channels, hidden_channels, kernel_size=3):
        super().__init__()

        padding = kernel_size // 2
        self.hidden_channels = hidden_channels

        self.conv = nn.Conv2d(
            input_channels + hidden_channels,
            4 * hidden_channels,
            kernel_size,
            padding=padding
        )

    def forward(self, x, h, c):
        combined = torch.cat([x, h], dim=1)
        gates = self.conv(combined)

        i, f, o, g = torch.chunk(gates, 4, dim=1)

        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)

        c_next = f * c + i * g
        h_next = o * torch.tanh(c_next)

        return h_next, c_next

In [22]:
class MovingMNISTPredictor(nn.Module):
    def __init__(self, input_channels=1, hidden_channels=64):
        super().__init__()

        self.hidden_channels = hidden_channels

        self.encoder = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, hidden_channels, kernel_size=3, padding=1),
            nn.ReLU()
        )

        self.convlstm = ConvLSTMCell(
            input_channels=hidden_channels,
            hidden_channels=hidden_channels
        )

        self.decoder = nn.Sequential(
            nn.Conv2d(hidden_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, input_channels, kernel_size=3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x, future_steps=10):
        """
        x: [B, T, C, H, W]
        returns: [B, future_steps, C, H, W]
        """

        B, T, C, H, W = x.shape
        device = x.device

        h = torch.zeros(B, self.hidden_channels, H, W, device=device)
        c = torch.zeros(B, self.hidden_channels, H, W, device=device)

        # Encode observed frames
        for t in range(T):
            z = self.encoder(x[:, t])
            h, c = self.convlstm(z, h, c)

        predictions = []

        # Autoregressive future prediction
        current_frame = x[:, -1]

        for _ in range(future_steps):
            z = self.encoder(current_frame)
            h, c = self.convlstm(z, h, c)

            pred_frame = self.decoder(h)
            predictions.append(pred_frame)

            current_frame = pred_frame

        return torch.stack(predictions, dim=1)

In [23]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import wandb

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

config = {
    "epochs": 50,
    "batch_size": 16,
    "learning_rate": 1e-3,
    "input_frames": 10,
    "future_steps": 10,
    "hidden_channels": 64,
    "patience": 5,
    "train_split": 0.8,
}


dataset = MovingObjectsDataset(X)

train_size = int(config["train_split"] * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config["batch_size"],
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config["batch_size"],
    shuffle=False
)

In [24]:
wandb.init(
    project="moving-mnist-prediction",
    name="convlstm-early-stopping",
    config=config,
)

wandb: Finishing previous runs because reinit is set to 'default'.
wandb: updating run metadata
wandb: uploading wandb-summary.json; uploading config.yaml
wandb: 🚀 View run convlstm-early-stopping at: https://wandb.ai/nec17/moving-mnist-prediction/runs/r6qtia0i
wandb: ⭐️ View project at: https://wandb.ai/nec17/moving-mnist-prediction
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260619_131607-r6qtia0i/logs
wandb: setting up run iiynonil
wandb: Tracking run with wandb version 0.27.2
wandb: Run data is saved locally in /Users/nel/Documents/learning_nlp/vision_transformer/wandb/run-20260619_131622-iiynonil
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run convlstm-early-stopping
wandb: ⭐️ View project at https://wandb.ai/nec17/moving-mnist-prediction
wandb: 🚀 View run at https://wandb.ai/nec17/moving-mnist-prediction/runs/iiynonil


In [25]:
model = MovingMNISTPredictor(
    input_channels=1,
    hidden_channels=config["hidden_channels"]
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config["learning_rate"]
)

loss_fn = nn.MSELoss()

wandb.watch(model, log="gradients", log_freq=100)

In [26]:
def prepare_batch(batch, device):
    if isinstance(batch, (tuple, list)):
        batch = batch[0]

    batch = batch.float().to(device)

    if batch.max() > 1:
        batch = batch / 255.0

    input_frames = batch[:, :config["input_frames"]]
    target_frames = batch[:, config["input_frames"]:]

    return input_frames, target_frames

In [27]:
def train_one_epoch(model, train_loader, optimizer, loss_fn, device, epoch):
    model.train()

    total_loss = 0.0

    for batch_idx, batch in enumerate(train_loader):
        input_frames, target_frames = prepare_batch(batch, device)

        pred_frames = model(
            input_frames,
            future_steps=config["future_steps"]
        )

        loss = loss_fn(pred_frames, target_frames)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        global_step = epoch * len(train_loader) + batch_idx

        wandb.log({
            "batch/train_loss": loss.item(),
            "epoch": epoch + 1,
        }, step=global_step)

        if batch_idx % 10 == 0:
            print(
                f"Epoch {epoch+1} | "
                f"Batch {batch_idx}/{len(train_loader)} | "
                f"Train Loss: {loss.item():.6f}"
            )

    avg_loss = total_loss / len(train_loader)
    return avg_loss

In [30]:
def evaluate(model, val_loader, loss_fn, device):
    model.eval()

    total_loss = 0.0

    with torch.no_grad():
        for batch in val_loader:
            input_frames, target_frames = prepare_batch(batch, device)

            pred_frames = model(
                input_frames,
                future_steps=config["future_steps"]
            )

            loss = loss_fn(pred_frames, target_frames)
            total_loss += loss.item()

    avg_loss = total_loss / len(val_loader)
    return avg_loss

In [ ]:
best_val_loss = float("inf")
epochs_without_improvement = 0

for epoch in range(config["epochs"]):
    # -------------------------
    # Training
    # -------------------------
    model.train()
    total_train_loss = 0.0
    total_train_mae = 0.0

    for batch_idx, batch in enumerate(train_loader):
        input_frames, target_frames = prepare_batch(batch, device)

        pred_frames = model(
            input_frames,
            future_steps=config["future_steps"]
        )

        loss = loss_fn(pred_frames, target_frames)
        mae = torch.mean(torch.abs(pred_frames - target_frames))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        total_train_mae += mae.item()

        global_step = epoch * len(train_loader) + batch_idx

        # Batch-level W&B logging
        wandb.log(
            {
                "batch/train_loss": loss.item(),
                "batch/train_mae": mae.item(),
                "epoch": epoch + 1,
            },
            step=global_step
        )

        if batch_idx % 10 == 0:
            print(
                f"Epoch {epoch+1}/{config['epochs']} | "
                f"Batch {batch_idx}/{len(train_loader)} | "
                f"Train Loss: {loss.item():.6f} | "
                f"Train MAE: {mae.item():.6f}"
            )

    avg_train_loss = total_train_loss / len(train_loader)
    avg_train_mae = total_train_mae / len(train_loader)

    # -------------------------
    # Validation
    # -------------------------
    model.eval()
    total_val_loss = 0.0
    total_val_mae = 0.0

    with torch.no_grad():
        for batch in val_loader:
            input_frames, target_frames = prepare_batch(batch, device)

            pred_frames = model(
                input_frames,
                future_steps=config["future_steps"]
            )

            loss = loss_fn(pred_frames, target_frames)
            mae = torch.mean(torch.abs(pred_frames - target_frames))

            total_val_loss += loss.item()
            total_val_mae += mae.item()

    avg_val_loss = total_val_loss / len(val_loader)
    avg_val_mae = total_val_mae / len(val_loader)

    # -------------------------
    # Check improvement
    # -------------------------
    improved = avg_val_loss < best_val_loss

    if improved:
        best_val_loss = avg_val_loss
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_loss": best_val_loss,
                "config": config,
            },
            "best_moving_mnist_model.pt"
        )

        wandb.save("best_moving_mnist_model.pt")

    else:
        epochs_without_improvement += 1

    # -------------------------
    # Epoch-level W&B logging
    # -------------------------
    epoch_step = (epoch + 1) * len(train_loader)

    wandb.log(
        {
            "epoch/train_loss": avg_train_loss,
            "epoch/train_mae": avg_train_mae,
            "epoch/val_loss": avg_val_loss,
            "epoch/val_mae": avg_val_mae,
            "epoch/best_val_loss": best_val_loss,
            "epoch/epochs_without_improvement": epochs_without_improvement,
            "epoch": epoch + 1,
        },
        step=epoch_step
    )

    print(
        f"Epoch {epoch+1}/{config['epochs']} | "
        f"Train Loss: {avg_train_loss:.6f} | "
        f"Train MAE: {avg_train_mae:.6f} | "
        f"Val Loss: {avg_val_loss:.6f} | "
        f"Val MAE: {avg_val_mae:.6f} | "
        f"Best Val Loss: {best_val_loss:.6f} | "
        f"No improvement: {epochs_without_improvement}/{config['patience']}"
    )

    # -------------------------
    # Early stopping
    # -------------------------
    if epochs_without_improvement >= config["patience"]:
        print("Early stopping triggered.")
        break

wandb.finish()

Epoch 1/50 | Batch 0/500 | Train Loss: 0.254903 | Train MAE: 0.501531
Epoch 1/50 | Batch 10/500 | Train Loss: 0.040380 | Train MAE: 0.050702
Epoch 1/50 | Batch 20/500 | Train Loss: 0.039935 | Train MAE: 0.046028
Epoch 1/50 | Batch 30/500 | Train Loss: 0.041823 | Train MAE: 0.048411
Epoch 1/50 | Batch 40/500 | Train Loss: 0.040343 | Train MAE: 0.047255
Epoch 1/50 | Batch 50/500 | Train Loss: 0.042808 | Train MAE: 0.049477
Epoch 1/50 | Batch 60/500 | Train Loss: 0.042057 | Train MAE: 0.048501
Epoch 1/50 | Batch 70/500 | Train Loss: 0.041605 | Train MAE: 0.048527
Epoch 1/50 | Batch 80/500 | Train Loss: 0.043361 | Train MAE: 0.050169
Epoch 1/50 | Batch 90/500 | Train Loss: 0.039272 | Train MAE: 0.045155
Epoch 1/50 | Batch 100/500 | Train Loss: 0.039933 | Train MAE: 0.046616
Epoch 1/50 | Batch 110/500 | Train Loss: 0.044846 | Train MAE: 0.051620
Epoch 1/50 | Batch 120/500 | Train Loss: 0.036886 | Train MAE: 0.043639
Epoch 1/50 | Batch 130/500 | Train Loss: 0.045545 | Train MAE: 0.052577
Epo

wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Epoch 1/50 | Train Loss: 0.044479 | Train MAE: 0.054330 | Val Loss: 0.042387 | Val MAE: 0.049097 | Best Val Loss: 0.042387 | No improvement: 0/5
Epoch 2/50 | Batch 0/500 | Train Loss: 0.042218 | Train MAE: 0.048869
Epoch 2/50 | Batch 10/500 | Train Loss: 0.050792 | Train MAE: 0.057851
Epoch 2/50 | Batch 20/500 | Train Loss: 0.043619 | Train MAE: 0.050244
Epoch 2/50 | Batch 30/500 | Train Loss: 0.042372 | Train MAE: 0.049381
Epoch 2/50 | Batch 40/500 | Train Loss: 0.047088 | Train MAE: 0.054218
Epoch 2/50 | Batch 50/500 | Train Loss: 0.043441 | Train MAE: 0.050176
Epoch 2/50 | Batch 60/500 | Train Loss: 0.044449 | Train MAE: 0.051573
Epoch 2/50 | Batch 70/500 | Train Loss: 0.041003 | Train MAE: 0.047475
Epoch 2/50 | Batch 80/500 | Train Loss: 0.040717 | Train MAE: 0.047365
Epoch 2/50 | Batch 90/500 | Train Loss: 0.038066 | Train MAE: 0.044496
Epoch 2/50 | Batch 100/500 | Train Loss: 0.040243 | Train MAE: 0.046763
Epoch 2/50 | Batch 110/500 | Train Loss: 0.047082 | Train MAE: 0.054617
Ep